In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

date_column = ['Flight Date']

birdstrikes_raw = pd.read_csv('data/birdstrikes.csv', parse_dates=date_column)
regions_raw = pd.read_csv('data/us_regions.csv')

birdstrikes = birdstrikes_raw.copy()
regions = regions_raw.copy()

print(birdstrikes.shape)
print(regions.shape)

(10000, 14)
(29, 2)


On the raw data:
- Why exactly 10000 rows? Has the data been cutoff? And if so, what is missing?
- Always refer to birdstrike-data as "reported..."
- No "id"
- Lots of zeros in the three cost columns: 'Cost Other', 'Cost Repair' and 'Cost Total $'
- IAS = 0 in rows where speed cannot not be 0
- Coyote in column 'Wildlife species' don't fit when counting birdstrikes
- 24 duplicates

Questions:
- When, in the flight and during the day, do the strikes happen? (starting question)
- What does a strike cost, and what does a cost of zero mean? (starting question)
- Does the number of reports over the years mean more strikes, or more reporting? (starting question)

### Dataset info - FAA aircraft wildlife strike reports
##### path: birdstrikes.csv
##### description: Records of reported wildlife strikes received by the U.S. FAA
##### schema:
| name | type | |:————————–|:——–| | Airport Name | string | | Aircraft Make Model | string | | Effect Amount of damage | string | | Flight Date | date | | Aircraft Airline Operator | string | | Origin State | string | | Phase of flight | string | | Wildlife Size | string | | Wildlife Species | string | | Time of day | string | | Cost Other | integer | | Cost Repair | integer | | Cost Total $ | integer | | Speed IAS in knots | integer |
##### sources:
| title | path | |:—————————–|:————————| | FAA Wildlife Strike Database | http://wildlife.faa.gov |
##### licenses:
| name | title | path | |:———|:————————|:——————————————| | other-pd | U.S. Government Dataset | https://resources.data.gov/open-licenses/ |

In [6]:
birdstrikes.dtypes

Airport Name                            str
Aircraft Make Model                     str
Effect Amount of damage                 str
Flight Date                  datetime64[us]
Aircraft Airline Operator               str
Origin State                            str
Phase of flight                         str
Wildlife Size                           str
Wildlife Species                        str
Time of day                             str
Cost Other                            int64
Cost Repair                           int64
Cost Total $                          int64
Speed IAS in knots                  float64
dtype: object

**Lots of 0s in cost columns :**

In [11]:
birdstrikes["Cost Other"].describe().round(2)

count      10000.00
mean         424.24
std        18298.22
min            0.00
25%            0.00
50%            0.00
75%            0.00
max      1565354.00
Name: Cost Other, dtype: float64

In [12]:
birdstrikes["Cost Repair"].describe().round(2)

count      10000.00
mean        3630.29
std        97869.97
min            0.00
25%            0.00
50%            0.00
75%            0.00
max      7043545.00
Name: Cost Repair, dtype: float64

In [13]:
birdstrikes["Cost Total $"].describe().round(2)

count      10000.00
mean        4054.53
std       102135.32
min            0.00
25%            0.00
50%            0.00
75%            0.00
max      7043545.00
Name: Cost Total $, dtype: float64

**Speed = 0 :**
There are speed = 0 birdstrikes where the aircraft is parked, these make sense, but there are others where the information is clearly wrong.
The false 0s needs to be distinguished before any aggregate calculations on speed.

In [15]:
birdstrikes["Speed IAS in knots"].describe()

count    7164.000000
mean      153.535176
std        43.518503
min         0.000000
25%       130.000000
50%       140.000000
75%       170.000000
max       350.000000
Name: Speed IAS in knots, dtype: float64

In [25]:
speed_zero = birdstrikes["Speed IAS in knots"] == 0

birdstrikes.loc[speed_zero].head(50)

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
276,ATLANTIC CITY INTL,F-16,NaN,1990-09-18,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,5218,5218,0.0
340,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
341,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
363,HONOLULU INTL ARPT,E-6,NaN,1990-10-24,MILITARY,Hawaii,Landing Roll,Medium,Unknown bird or bat,Day,0,0,0,0.0
385,BARKSDALE AIR FORCE BASE ARPT,B-52G,NaN,1990-10-27,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Day,0,0,0,0.0
562,BARKSDALE AIR FORCE BASE ARPT,B-52G,NaN,1991-05-02,MILITARY,Louisiana,Approach,Small,Cattle egret,Night,0,0,0,0.0
614,BARKSDALE AIR FORCE BASE ARPT,A-10A,NaN,1991-05-22,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Dawn,0,2711,2711,0.0
954,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1991-11-06,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Day,0,0,0,0.0
1327,CHICAGO O'HARE INTL ARPT,B-767-300,NaN,1992-07-16,UNITED AIRLINES,Illinois,Take-off run,Medium,Unknown bird - medium,Day,0,0,0,0.0
1395,BARKSDALE AIR FORCE BASE ARPT,B-52G,NaN,1992-08-13,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Night,0,0,0,0.0


**Coyote as 'Wildlife Species' :** The 26 coyotes are certainly wildlife, but they are not birds.

In [26]:
birdstrikes["Wildlife Species"].value_counts()

Wildlife Species
Unknown bird - small     3572
Unknown bird - medium    3372
Unknown bird or bat       629
Unknown bird - large      436
European starling         319
Rock pigeon               232
Mourning dove             217
Canada goose              190
Red-tailed hawk           100
Ring-billed gull           87
American kestrel           83
Pacific golden-plover      65
Killdeer                   64
Mallard                    63
American robin             62
Herring gull               52
Barn owl                   47
American crow              41
Barn swallow               38
White-tailed deer          34
Turkey vulture             33
Eastern meadowlark         30
Laughing gull              29
Coyote                     26
Great blue heron           25
Zebra dove                 25
Horned lark                22
Western meadowlark         19
Cattle egret               16
Tree swallow               14
Cliff swallow              13
Blackbirds                 10
Meadowlarks            

**Time of day, Wildlife Size, Phase of flight, Origin State, Flight Date:** Categories look alright

In [37]:
birdstrikes["Time of day"].value_counts(dropna=False)

Time of day
Day      5624
Night    3363
Dusk      584
Dawn      429
Name: count, dtype: int64

In [38]:
birdstrikes["Wildlife Size"].value_counts(dropna=False)

Wildlife Size
Small     4910
Medium    4346
Large      744
Name: count, dtype: int64

In [39]:
birdstrikes["Phase of flight"].value_counts(dropna=False)

Phase of flight
Approach        4619
Climb           1956
Take-off run    1592
Landing Roll    1405
Descent          399
Taxi              18
Parked            11
Name: count, dtype: int64

In [40]:
birdstrikes["Origin State"].value_counts(dropna=False)

Origin State
Texas             1495
California         890
Louisiana          618
Tennessee          569
Kentucky           535
Pennsylvania       514
Illinois           505
DC                 475
New York           391
Missouri           376
Hawaii             352
New Jersey         351
North Carolina     269
Florida            246
Oregon             245
South Carolina     242
Utah               236
Georgia            211
Ohio               210
Maryland           201
Colorado           187
Massachusetts      146
Indiana            144
Nebraska           118
Arizona            111
Minnesota          103
Washington         103
Oklahoma            83
Michigan            74
Name: count, dtype: int64

In [50]:
print(birdstrikes["Flight Date"].isna().sum())
birdstrikes["Flight Date"].describe()

0


count                         10000
mean     1997-03-20 11:22:16.320000
min             1990-01-08 00:00:00
25%             1994-04-30 18:00:00
50%             1997-08-29 00:00:00
75%             2000-05-21 06:00:00
max             2002-07-25 00:00:00
Name: Flight Date, dtype: object

In [ ]:
birdstrikes["Aircraft Make Model"].value_counts(dropna=False)

**Effect Amount of damage**

- 8939 NaN
- "B" and "C" seems to be of a different nomenclature than the other values

In [53]:
birdstrikes["Effect Amount of damage"].value_counts(dropna=False)

Effect Amount of damage
NaN            8939
Minor           549
Substantial     311
Medium          186
C                14
B                 1
Name: count, dtype: int64

**Aircraft Airline Operators**

- There are 72 UNKNOWN

- GOVERNMENT (15) and MILITARY (829) could be overlapping categories

- PRIVATELY OWNED (36), BUSINESS (371) could be overlapping categories



In [43]:
birdstrikes["Aircraft Airline Operator"].value_counts(dropna=False)

Aircraft Airline Operator
AMERICAN AIRLINES                 2171
US AIRWAYS*                       1084
DELTA AIR LINES                    865
SOUTHWEST AIRLINES                 844
MILITARY                           829
UNITED AIRLINES                    534
BUSINESS                           371
FEDEX EXPRESS                      365
AMERICA WEST AIRLINES              275
NORTHWEST AIRLINES                 256
UPS AIRLINES                       223
AMERICAN EAGLE AIRLINES            223
CONTINENTAL AIRLINES               187
ALOHA AIRLINES                     184
TRANS WORLD AIRLINES               163
ATLANTIC COAST AIRLINES            138
COMAIR AIRLINES                    113
EXPRESSJET (CONTINENTAL EXPRS)     106
ALASKA AIRLINES                    102
HORIZON AIR                        102
HAWAIIAN AIR                        97
ABX AIR                             76
UNKNOWN                             72
AIR CANADA                          69
SKYWEST AIRLINES                    68

**No empty columns:**

In [14]:
empty = birdstrikes.columns[birdstrikes.isna().all()]

print(len(empty))
print(list(empty))

0
[]


**24 duplicates**

In [27]:
print(birdstrikes.duplicated().sum())

24


In [29]:
birdstrikes[birdstrikes.duplicated(keep=False)].head(10)

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
340,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
341,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
1113,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-14,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
1114,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-14,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
1131,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-23,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,140.0
1132,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-23,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,140.0
1267,GREATER PITTSBURGH,DC-9-30,NaN,1992-06-15,US AIRWAYS*,Pennsylvania,Climb,Small,Unknown bird - small,Day,0,0,0,150.0
1268,GREATER PITTSBURGH,DC-9-30,NaN,1992-06-15,US AIRWAYS*,Pennsylvania,Climb,Small,Unknown bird - small,Day,0,0,0,150.0
1813,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1993-04-28,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,180.0
1814,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1993-04-28,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,180.0
